# PatchVision-X: training notebook

A compact Vision Transformer trained **from scratch** on CUB-200-2011 (5,394 train / 600 val / 5,794 test images).
All model code lives in the `patchvision/` package next to this notebook; this notebook only configures and starts training.

**PatchVision-X** = GvT's sparse graph attention (Sci. Rep. 2025) + MSCViT's local-feature block and multi-scale attention (arXiv 2501.06040),
realised as *multi-scale graph attention*, plus four small-data regularisers: bounding-box attention loss, EMA weights, flip-consistency loss, box-guided crops.

**Everything is saved under `results/<experiment>/seed<seed>/`**: the step-by-step `train_log.txt`, `steps.csv`, `history.csv`, curves, attention snapshots, checkpoints, test metrics, confusion matrix, calibration plot.

**How to run**
1. Run the first cell with `SMOKE_TEST = True` (2 epochs on synthetic images, a few minutes) to check the whole pipeline on your machine.
2. Set `SMOKE_TEST = False` and run all cells for the real run. Interrupted? Run all again: it resumes from `last.pt`.
3. After code edits in `patchvision/`, restart the kernel.

In [ ]:
import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))          # the folder that contains patchvision/

import torch, pandas as pd
from IPython.display import Image as show_image, display

from patchvision.config import PathsConfig, get_experiment, ablation_suite
from patchvision.data import make_synthetic_dataset, load_data_info, build_loaders
from patchvision.experiment import dry_run, run_experiment, run_suite, summarize_results
from patchvision import visualize as viz

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY - training will be very slow")

## 1. Settings

In [ ]:
# ================================ USER SETTINGS ================================
SMOKE_TEST       = True          # True: 2 epochs on synthetic data to test the pipeline. Set False for the real run.
EXPERIMENT       = "full"        # any key of ablation_suite(): "full", "base", "arch_x", "full-ema", ...
SEEDS            = [42]          # use [42, 43, 44] for mean +/- std over seeds
NUM_WORKERS      = 4             # data-loading processes (use 0 if you ever hit a multiprocessing error)
STEP_PRINT_EVERY = 1             # 1 = print EVERY training step (the log file always has every step)
RESUME           = True          # continue from last.pt if a run was interrupted

MODEL_OVERRIDES  = {}            # e.g. {"embed_dim": 384, "depth": 6, "heads": 8}
TRAIN_OVERRIDES  = {}            # e.g. {"epochs": 200, "lr": 3e-4, "bbox_weight": 0.2, "early_stop_patience": 40}
# ===============================================================================

PATHS = PathsConfig(root=".")    # DataPreprocessing_Results/, CUB_200_2011/ and results/ are looked up here
if SMOKE_TEST:
    PATHS = make_synthetic_dataset("smoke_data", n_classes=8, n_train=12, n_val=2, n_test=4)
    TRAIN_OVERRIDES = dict(epochs=2, warmup_epochs=1, batch_size=16, snapshot_every=1, clean_eval_every=1,
                           clean_eval_size=16, early_stop_patience=0, weights_every=0)
    NUM_WORKERS = min(NUM_WORKERS, 2)
    print(">>> SMOKE TEST: synthetic data in ./smoke_data, 2 epochs <<<")
print("experiments available:", list(ablation_suite()))
print("results will be written to:", PATHS.results)

## 2. Data check: augmented images with their bird box (green). The box must stay on the bird.

In [ ]:
exp = get_experiment(EXPERIMENT, SEEDS[0], MODEL_OVERRIDES, TRAIN_OVERRIDES)
info = load_data_info(PATHS)
print(f"splits {info.splits} | classes {info.num_classes} | normalisation mean {[round(m,3) for m in info.mean]}")
ds, _ = build_loaders(info, exp.train, exp.seed, num_workers=0, pin_memory=False)
(PATHS.results / "_checks").mkdir(parents=True, exist_ok=True)
viz.augmentation_preview(ds["train"], info.mean, info.std, PATHS.results / "_checks" / "augmentation_check.png",
                         n=12, show=True, title=f"tier '{exp.train.aug_tier}' + box-guided crop (p={exp.train.box_crop_prob if exp.train.use_box_crop else 0})")

## 3. Dry run: parameters, FLOPs, one real batch forward/backward, gradient check

In [ ]:
report = dry_run(exp, PATHS)

## 4. Train

Every step prints one line: loss split into CE / bbox / consistency, accuracy, attention-inside-box, learning rate, gradient norm, gate sparsity, images/s, GPU memory, ETA.
After every epoch you get TRAIN / VAL / **CLEAN** (train images without augmentation, the honest overfitting gauge) / ATTN / TIME / BEST.

In [ ]:
rows = []
for seed in SEEDS:
    print(f"\n######################## {EXPERIMENT} | seed {seed} ########################")
    run_exp = get_experiment(EXPERIMENT, seed, MODEL_OVERRIDES, TRAIN_OVERRIDES)
    rows.append(run_experiment(run_exp, PATHS, num_workers=NUM_WORKERS,
                               step_print_every=STEP_PRINT_EVERY, resume=RESUME))
results = pd.DataFrame(rows)
results

## 5. Results

In [ ]:
display(summarize_results(PATHS))
run_dir = Path(rows[-1]["run_dir"])
print("all files of the last run are in:", run_dir)
for name in ("curves.png", "test_attention_gallery.png", "test_confusion_matrix.png", "test_reliability.png"):
    if (run_dir / name).exists():
        print(name)
        display(show_image(filename=str(run_dir / name)))

## 6. Ablation suite (optional, long)

Runs several variants and writes `results/ablation_table.csv` / `.md` (mean +/- std over seeds).
Finished runs are skipped, so you can run this cell again after an interruption.
`base` = GvT-style anchor; `arch_x` = PatchVision-X architecture without regularisers; `full` = everything; `full-<x>` = leave one component out.

In [ ]:
RUN_ABLATIONS = False
SUITE = ["base", "lfe", "multiscale", "arch_x", "arch_x+bbox", "arch_x+ema", "arch_x+cons", "arch_x+boxcrop", "full"]
if RUN_ABLATIONS:
    run_suite(SUITE, SEEDS, PATHS, MODEL_OVERRIDES, TRAIN_OVERRIDES, num_workers=NUM_WORKERS,
              step_print_every=STEP_PRINT_EVERY, resume=RESUME)
    display(summarize_results(PATHS))

## 7. Live monitor (read-only)

A running training cell blocks its notebook kernel, so run this from a **second notebook/kernel**
(or from a terminal: `python watch_training.py full 42`). It refreshes every 2 s with a progress bar for the current epoch
and a table of the latest epochs, and stops by itself when the run has finished. Interrupting it never affects the training.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from patchvision.monitor import watch

watch(Path("results") / "full" / "seed42", interval=2.0, rows=15)     # change "full"/"seed42" for another run